In [3]:
# Testing the Brunner–Munzel Variance Formula

'''
We verify the paper's local-linear-approximation variance formula:

    Var_hat(qhat) = s_x^2 / (n m^2) + s_y^2 / (n^2 m)

where s_x^2 and s_y^2 are sample variances of **within-group ranks**.
'''

## Plan
'''
For each experiment:
1. Repeat 1000 times: draw X (n pts), Y (m pts), compute qhat and Var_hat(qhat).
2. Compare empirical Var(qhat) to mean Var_hat(qhat). Ratio should be ≈ 1.
3. Check bias: mean(qhat) vs true q.
4. Check CI coverage (normal and t with Welch df). Should be ≈ 0.95.
5. Under H0, check rejection rate at α = 0.05. Should be ≈ 0.05.
'''

## Experiments
'''
- **H0:** N(0,1) vs N(0,1), n = m = 100 → q = 0.5
- **H1:** N(0,1) vs N(0.5,1), n = m = 100 → q = Φ(0.5/√2)
- **Small n:** same two, n = m = 10 → shows t beats normal
- **Skewed:** Exp(1) vs Exp(1), n = m = 30 → q = 0.5
'''

'\n- **H0:** N(0,1) vs N(0,1), n = m = 100 → q = 0.5\n- **H1:** N(0,1) vs N(0.5,1), n = m = 100 → q = Φ(0.5/√2)\n- **Small n:** same two, n = m = 10 → shows t beats normal\n- **Skewed:** Exp(1) vs Exp(1), n = m = 30 → q = 0.5\n'

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(20260928)
pd.set_option("display.float_format", lambda v: f"{v: .4f}")

In [5]:
## Core BM machinery
'''
- `heaviside_matrix(x, y)`: builds A with A_ij = H(y_j − x_i), H(0) = 0.5.
- `qhat_from_A(A)`: mean of all entries.
- `within_group_ranks(v)`: average ranks of v within itself (handles ties).
- `bm_variance(x, y)`: local-linear-approx variance using **within-group ranks**.
- `bm_welch_df(x, y)`: Satterthwaite–Smith–Welch df for the t-approximation.
'''
    
def heaviside_matrix(x, y):
    diff = y[None, :] - x[:, None]
    return np.where(diff > 0, 1.0, np.where(diff < 0, 0.0, 0.5))

def qhat_from_A(A):
    return A.mean()

def within_group_ranks(v):
    return stats.rankdata(v, method="average")

def bm_variance(x, y):
    n, m = len(x), len(y)
    sx2 = within_group_ranks(x).var(ddof=1)
    sy2 = within_group_ranks(y).var(ddof=1)
    return sx2 / (n * m**2) + sy2 / (n**2 * m)

def bm_statistic(x, y):
    q = qhat_from_A(heaviside_matrix(x, y))
    return q, bm_variance(x, y)

def bm_welch_df(x, y):
    n, m = len(x), len(y)
    sx2 = within_group_ranks(x).var(ddof=1)
    sy2 = within_group_ranks(y).var(ddof=1)
    vx = sx2 / (n * m**2)
    vy = sy2 / (n**2 * m)
    return (vx + vy)**2 / (vx**2 / (n - 1) + vy**2 / (m - 1))

In [6]:
## True q

'''
For independent normals:
    q = P(Y > X) = Φ( (μ_y − μ_x) / √(σ_x^2 + σ_y^2) )

For independent exponentials with rates λ_x, λ_y:
    q = P(Y > X) = λ_x / (λ_x + λ_y)

No midpoint — just the standardized difference.
'''

def true_q_normal(mu_x, sd_x, mu_y, sd_y):
    return stats.norm.cdf((mu_y - mu_x) / np.sqrt(sd_x**2 + sd_y**2))

def true_q_exp(rate_x, rate_y):
    return rate_x / (rate_x + rate_y)

In [7]:
## Simulation engine

'''
`run_experiment(n, m, dist_x, dist_y, n_rep)` returns arrays of:
- qhat
- Var_hat(qhat)
- W = (qhat − 0.5) / sqrt(Var_hat)
- Welch df

`summarize(...)` produces a one-row table with:
- mean qhat, bias
- empirical Var vs mean Var_hat, ratio
- CI coverage (normal and t) at 95%
- rejection rate at α = 0.05
'''


def run_experiment(n, m, dist_x, dist_y, n_rep=1000):
    qs = np.empty(n_rep); vs = np.empty(n_rep)
    ws = np.empty(n_rep); dfs = np.empty(n_rep)
    for r in range(n_rep):
        x = dist_x(n, rng); y = dist_y(m, rng)
        q, v = bm_statistic(x, y)
        qs[r] = q; vs[r] = v
        ws[r] = (q - 0.5) / np.sqrt(v)
        dfs[r] = bm_welch_df(x, y)
    return dict(qs=qs, vs=vs, ws=ws, dfs=dfs)

def summarize(res, true_q, label, alpha=0.05):
    qs, vs, ws, dfs = res["qs"], res["vs"], res["ws"], res["dfs"]
    z = stats.norm.ppf(1 - alpha/2)
    tcrit = stats.t.ppf(1 - alpha/2, dfs)
    cov_n = np.mean((qs - z*np.sqrt(vs) <= true_q) & (true_q <= qs + z*np.sqrt(vs)))
    cov_t = np.mean((qs - tcrit*np.sqrt(vs) <= true_q) & (true_q <= qs + tcrit*np.sqrt(vs)))
    return pd.Series({
        "experiment": label,
        "true q": true_q,
        "mean qhat": qs.mean(),
        "bias": qs.mean() - true_q,
        "empirical Var": qs.var(ddof=1),
        "mean Var_hat": vs.mean(),
        "ratio": vs.mean() / qs.var(ddof=1),
        "cov normal": cov_n,
        "cov t": cov_t,
        "rej normal": np.mean(np.abs(ws) > z),
        "rej t": np.mean(np.abs(ws) > tcrit),
    })

In [8]:
## Experiments

'''
Define each case as (n, m, dist_x, dist_y, true_q, label).
'''

def normal(mu, sd):
    return lambda n, rng: rng.normal(mu, sd, size=n)

def expo(rate):
    return lambda n, rng: rng.exponential(1/rate, size=n)

q_H1 = true_q_normal(0, 1, 0.5, 1)

experiments = [
    (100, 100, normal(0,1), normal(0,1),   0.5,   "H0  n=m=100"),
    (100, 100, normal(0,1), normal(0.5,1), q_H1,  "H1  n=m=100"),
    (10,  10,  normal(0,1), normal(0,1),   0.5,   "H0  n=m=10"),
    (10,  10,  normal(0,1), normal(0.5,1), q_H1,  "H1  n=m=10"),
    (30,  30,  expo(1),     expo(1),       0.5,   "H0  Exp n=m=30"),
]

In [10]:
## Run the simulations

#1000 replications per experiment. The summary table reports all diagnostics.

rows = []
for n, m, dx, dy, tq, label in experiments:
    res = run_experiment(n, m, dx, dy, n_rep=1000)
    rows.append(summarize(res, tq, label))

summary = pd.DataFrame(rows).set_index("experiment")
summary

,true q,mean qhat,bias,empirical Var,mean Var_hat,ratio,cov normal,cov t,rej normal,rej t
experiment,,,,,,,,,,
H0 n=m=100,0.5000,0.5010,0.0010,0.0016,0.0017,1.0663,0.9570,0.9590,0.0430,0.0410
H1 n=m=100,0.6382,0.6380,-0.0002,0.0015,0.0017,1.1024,0.9510,0.9520,0.9300,0.9260
H0 n=m=10,0.5000,0.4976,-0.0024,0.0182,0.0183,1.0067,0.9480,0.9620,0.0520,0.0380
H1 n=m=10,0.6382,0.6339,-0.0043,0.0152,0.0183,1.2067,0.9680,0.9820,0.1500,0.1120
H0 Exp n=m=30,0.5000,0.5033,0.0033,0.0057,0.0057,0.9984,0.9480,0.9560,0.0520,0.0440


In [12]:
## How to read the table

'''
- **bias** ≈ 0 → qhat is unbiased.
- **ratio** ≈ 1 → variance formula is accurate.
- **cov normal**, **cov t** ≈ 0.95 → CIs have nominal coverage.
  - At small n, cov t should beat cov normal.
- **rej normal**, **rej t** ≈ 0.05 under H0 → Type I error is controlled.
  - At small n, rej t should beat rej normal.
'''

'\n- **bias** ≈ 0 → qhat is unbiased.\n- **ratio** ≈ 1 → variance formula is accurate.\n- **cov normal**, **cov t** ≈ 0.95 → CIs have nominal coverage.\n  - At small n, cov t should beat cov normal.\n- **rej normal**, **rej t** ≈ 0.05 under H0 → Type I error is controlled.\n  - At small n, rej t should beat rej normal.\n'